In [0]:
-- DATA DA FOTO 2026-06-01

SELECT IdCliente,
       count(distinct date(DtCriacao)) AS frequecia,
       count(*) AS frequeciaTransacoes,
       sum(case when QtdePontos > 0 then qtdePontos else 0 end) AS valor,
       min(date_diff('2026-06-01', DtCriacao)) AS recencia

FROM workspace.tmw_loyalty.transacoes

WHERE DtCriacao < '2026-06-01'
AND DtCriacao >= '2026-06-01' - interval 28 days

GROUP BY ALL


Databricks visualization. Run in Databricks to view.

In [0]:
%python

from sklearn import cluster
from sklearn import preprocessing

kmeans = cluster.KMeans(n_clusters=8)

min_max = preprocessing.MinMaxScaler()

df = _sqldf.toPandas()

df = df[df['IdCliente']!='163022e8-12b8-486f-8604-57d8fa0ed7e1']


X = df[['frequecia', 'valor']]
X = min_max.fit_transform(X)

kmeans.fit(X)

df['cluster'] = kmeans.predict(X)
sdf = spark.createDataFrame(df)
sdf.display()

Databricks visualization. Run in Databricks to view.

In [0]:
WITH tb_user_eps AS (

    SELECT idUsuario,
        descSlugCurso,
        count(descSlugCursoEpisodio) as qtdeEP,
        max(dtCriacao) AS last_dt,
        MIN(dtCriacao) AS first_dt
    FROM workspace.tmw_education.cursos_episodios_completos
    GROUP BY ALL
),

tb_cursos AS (

    SELECT descSlugCurso,
        count(descEpisodio) as qtd_ep_curso

    FROM workspace.tmw_education.cursos_episodios
    GROUP BY ALL

)

SELECT *,
        t1.qtdeEP / t2.qtd_ep_curso as pct_curso_completo,
        date_diff(last_dt,first_dt) as diff_tempo

FROM tb_user_eps AS t1
LEFT JOIN tb_cursos as t2
ON t1.descSlugCurso = t2.descSlugCurso